# Step 5D.2 — незалежний аудит результатів (без ретюнінгу)

Мета цього notebook: перевірити реальні server-результати Step 5D.2 (CIC/UNSW, F0/F1),
спираючись, наскільки можливо, на **збережені артефакти**, а не на in-memory змінні того
самого прогону. Це НЕ новий підбір гіперпараметрів і НЕ зміна frozen-протоколу.

Оскільки Step 5D.2 зберіг лише агреговані таблиці (`step5d2_f0_f1_results_h_flow.csv`,
`step5d2_delta_ap.csv`, `step5d2_topk_*.csv`), а не сирі test-прогнози чи файли моделей,
повністю "апаратна" перевірка (перерахунок AP напряму з raw scores того самого прогону)
неможлива — це сам по собі важливий висновок аудиту (розділ 0 нижче).

Тому аудит складається з ДВОХ незалежних частин:

1. **Internal-consistency audit** (розділи 1-3) — перевірка збережених CSV-артефактів
   на арифметичну узгодженість між собою і з раніше замороженими числами Step 5C.2/5D.1,
   без жодного повторного навчання моделі.
2. **Independent reproduction** (розділи 4-7) — повторне ВІДТВОРЕННЯ (не підбір) frozen
   LightGBM-протоколу з тим самим seed/params у свіжому kernel-стані, з незалежним
   перерахунком AP/ROC-AUC/Top-K та звіркою проти збережених CSV з допустимим допуском.
   Цього разу сирі test-прогнози й ідентифікатори рядків додатково зберігаються, щоб
   майбутні перевірки могли бути повністю артефакт-базованими без повторного навчання.


In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"
ENRICHED_DIR = Path("/srv/ids_research/data/enriched")

print("RESULTS_DIR:", RESULTS_DIR, "exists:", RESULTS_DIR.exists())
print("ENRICHED_DIR:", ENRICHED_DIR, "exists:", ENRICHED_DIR.exists())


RESULTS_DIR: /home/ivan_r_opirskyy/notebooks/2 article/release/results exists: True
ENRICHED_DIR: /srv/ids_research/data/enriched exists: True


## 0. Інвентаризація збережених артефактів Step 5D.2

In [2]:
expected_artifacts = [
    "step5d2_f0_f1_results_h_flow.csv",
    "step5d2_delta_ap.csv",
    "step5d2_topk_CIC_F0.csv",
    "step5d2_topk_CIC_F1.csv",
    "step5d2_topk_UNSW_F0.csv",
    "step5d2_topk_UNSW_F1.csv",
]

print("Наявність очікуваних артефактів Step 5D.2:")
missing = []
for name in expected_artifacts:
    p = RESULTS_DIR / name
    ok = p.exists()
    print(f"  {'OK  ' if ok else 'ВІДСУТНІЙ'}  {name}")
    if not ok:
        missing.append(name)

assert not missing, (
    f"Відсутні очікувані артефакти: {missing}. Без них частина 1-3 (internal-consistency "
    f"audit) не може виконатись — спочатку перевірте RESULTS_DIR / повторно збережіть "
    f"результати Step 5D.2 (комірка 16 того notebook)."
)

print(
    "\nℹ Ці файли містять ЛИШЕ агреговані метрики (results/delta/top-K), а НЕ сирі "
    "test-прогнози чи файли моделей. Тому пункти 1-3 нижче є артефакт-базованою перевіркою "
    "внутрішньої узгодженості, а не незалежним перерахунком AP з нуля — це чесно "
    "задокументовано, а не приховано."
)


Наявність очікуваних артефактів Step 5D.2:
  OK    step5d2_f0_f1_results_h_flow.csv
  OK    step5d2_delta_ap.csv
  OK    step5d2_topk_CIC_F0.csv
  OK    step5d2_topk_CIC_F1.csv
  OK    step5d2_topk_UNSW_F0.csv
  OK    step5d2_topk_UNSW_F1.csv

ℹ Ці файли містять ЛИШЕ агреговані метрики (results/delta/top-K), а НЕ сирі test-прогнози чи файли моделей. Тому пункти 1-3 нижче є артефакт-базованою перевіркою внутрішньої узгодженості, а не незалежним перерахунком AP з нуля — це чесно задокументовано, а не приховано.


## 1. Завантаження збережених результатів (без жодного in-memory стану Step 5D.2)

In [3]:
results_saved = pd.read_csv(RESULTS_DIR / "step5d2_f0_f1_results_h_flow.csv")
delta_saved = pd.read_csv(RESULTS_DIR / "step5d2_delta_ap.csv")

topk_saved = {}
for dataset_name in ["CIC", "UNSW"]:
    for feature_set_name in ["F0", "F1"]:
        topk_saved[(dataset_name, feature_set_name)] = pd.read_csv(
            RESULTS_DIR / f"step5d2_topk_{dataset_name}_{feature_set_name}.csv"
        )

print("results_saved:")
display(results_saved)
print("\ndelta_saved:")
display(delta_saved)


results_saved:


,dataset,feature_set,n_features,best_iteration,val_ap,val_roc_auc,test_ap,test_roc_auc,test_precision,test_recall,test_f1
0,CIC,F0,77,4,0.778550,0.823532,0.710369,0.740553,0.000000,0.000000,0.000000
1,CIC,F1,84,118,0.676896,0.765135,0.974028,0.982162,0.997403,0.604780,0.752984
2,UNSW,F0,41,33,0.964633,0.997017,0.973652,0.997671,0.778539,0.999842,0.875421
3,UNSW,F1,46,290,0.979776,0.997882,0.983969,0.998247,0.779184,0.998028,0.875132



delta_saved:


,dataset,F0_AP,F1_AP,delta_AP,F0_ROC_AUC,F1_ROC_AUC,delta_ROC_AUC
0,CIC,0.710369,0.974028,0.263659,0.740553,0.982162,0.241609
1,UNSW,0.973652,0.983969,0.010317,0.997671,0.998247,0.000576


## 2. Internal-consistency audit #1 — ΔAP і ΔROC-AUC перераховані незалежно з results_saved

`delta_saved` було збережено ОКРЕМИМ файлом у тому самому прогоні. Тут ΔAP/ΔROC-AUC
перераховуються заново напряму з `results_saved` (інший файл) і звіряються з `delta_saved` —
це ловить помилку копіювання/розсинхронізації між двома артефактами, якщо вона є.

In [4]:
results_idx = results_saved.set_index(["dataset", "feature_set"])

recomputed_delta_rows = []
for dataset_name in ["CIC", "UNSW"]:
    ap_f0 = results_idx.loc[(dataset_name, "F0"), "test_ap"]
    ap_f1 = results_idx.loc[(dataset_name, "F1"), "test_ap"]
    roc_f0 = results_idx.loc[(dataset_name, "F0"), "test_roc_auc"]
    roc_f1 = results_idx.loc[(dataset_name, "F1"), "test_roc_auc"]
    recomputed_delta_rows.append(dict(
        dataset=dataset_name,
        F0_AP=ap_f0, F1_AP=ap_f1, delta_AP=ap_f1 - ap_f0,
        F0_ROC_AUC=roc_f0, F1_ROC_AUC=roc_f1, delta_ROC_AUC=roc_f1 - roc_f0,
    ))
recomputed_delta = pd.DataFrame(recomputed_delta_rows).set_index("dataset")

print("Перераховано незалежно з results_saved:")
display(recomputed_delta)

delta_saved_idx = delta_saved.set_index("dataset")
for dataset_name in ["CIC", "UNSW"]:
    for col in ["delta_AP", "delta_ROC_AUC"]:
        a = float(recomputed_delta.loc[dataset_name, col])
        b = float(delta_saved_idx.loc[dataset_name, col])
        assert abs(a - b) < 1e-9, (
            f"{dataset_name}.{col}: перерахунок={a:.6f} != збережено у delta CSV={b:.6f}"
        )
print("\n✅ ΔAP і ΔROC-AUC у step5d2_delta_ap.csv точно узгоджені з step5d2_f0_f1_results_h_flow.csv")


Перераховано незалежно з results_saved:


,F0_AP,F1_AP,delta_AP,F0_ROC_AUC,F1_ROC_AUC,delta_ROC_AUC
dataset,,,,,,
CIC,0.710369,0.974028,0.263659,0.740553,0.982162,0.241609
UNSW,0.973652,0.983969,0.010317,0.997671,0.998247,0.000576



✅ ΔAP і ΔROC-AUC у step5d2_delta_ap.csv точно узгоджені з step5d2_f0_f1_results_h_flow.csv


## 3. Internal-consistency audit #2 — Top-K таблиці узгоджені з собою і з results_saved

Для кожної (dataset, feature_set) перевіряємо:

- `K` відповідає заявленій частці `top_pct` від розміру test-вибірки (розмір test виводиться
  з `K / top_pct`, оскільки сирий test-набір тут не перезавантажується);
- TP-рахунок, неявно закодований у `precision * K`, є цілим числом (з точністю до похибки
  округлення double) — нецілий TP означає, що precision/K або сам topk-розрахунок пошкоджені;
- той самий TP-рахунок, отриманий через `recall * n_pos` (з `n_pos`, виведеним з
  найбільшого K), збігається з TP, отриманим через `precision * K` — це перевірка того,
  що precision і recall у збереженому файлі внутрішньо узгоджені одне з одним;
- монотонність: recall не спадає зі зростанням K (кожен наступний top-K є надмножиною
  попереднього за конструкцією `np.argsort`).

In [5]:
def audit_topk_table(name_pair, df):
    dataset_name, feature_set_name = name_pair
    print(f"\n----- {dataset_name}-{feature_set_name} -----")

    # n_pos_test виводимо з рядка з найбільшим K (найбільша, отже, найточніша оцінка)
    row_max_k = df.loc[df["K"].idxmax()]
    implied_n_pos = row_max_k["recall"] and (row_max_k["precision"] * row_max_k["K"]) / row_max_k["recall"]

    prev_recall = -1.0
    for _, row in df.iterrows():
        tp_from_precision = row["precision"] * row["K"]
        tp_from_recall = row["recall"] * implied_n_pos if implied_n_pos else np.nan

        tp_rounded = round(tp_from_precision)
        assert abs(tp_from_precision - tp_rounded) < 1e-6, (
            f"{name_pair} top_pct={row['top_pct']}: precision*K={tp_from_precision:.6f} "
            f"не є цілим числом -> TP не є консистентним лічильником"
        )
        if implied_n_pos:
            assert abs(tp_from_precision - tp_from_recall) < 0.51, (
                f"{name_pair} top_pct={row['top_pct']}: TP з precision={tp_from_precision:.3f} "
                f"vs TP з recall={tp_from_recall:.3f} -- розсинхронізовані"
            )

        assert row["recall"] >= prev_recall - 1e-9, (
            f"{name_pair} top_pct={row['top_pct']}: recall={row['recall']:.6f} "
            f"менший за recall попереднього (меншого) K -- порушення монотонності"
        )
        prev_recall = row["recall"]

        print(
            f"  top_pct={row['top_pct']:>5}  K={int(row['K']):>7,}  "
            f"precision={row['precision']:.6f}  recall={row['recall']:.6f}  "
            f"TP(round)={int(tp_rounded)}"
        )

    print(f"  ✅ внутрішньо узгоджено; implied n_pos_test (з найбільшого K) ≈ {implied_n_pos:.1f}"
          if implied_n_pos else "  ⚠ implied n_pos_test не визначено (recall=0 на найбільшому K)")
    return implied_n_pos


implied_n_pos_by_model = {}
for key, df in topk_saved.items():
    implied_n_pos_by_model[key] = audit_topk_table(key, df)



----- CIC-F0 -----
  top_pct= 0.1%  K=    429  precision=1.000000  recall=0.003202  TP(round)=429
  top_pct= 0.5%  K=  2,146  precision=1.000000  recall=0.016017  TP(round)=2146
  top_pct= 1.0%  K=  4,291  precision=1.000000  recall=0.032026  TP(round)=4291
  top_pct= 5.0%  K= 21,457  precision=1.000000  recall=0.160144  TP(round)=21457
  ✅ внутрішньо узгоджено; implied n_pos_test (з найбільшого K) ≈ 133986.0

----- CIC-F1 -----
  top_pct= 0.1%  K=    429  precision=0.997669  recall=0.003194  TP(round)=428
  top_pct= 0.5%  K=  2,146  precision=0.999534  recall=0.016009  TP(round)=2145
  top_pct= 1.0%  K=  4,291  precision=0.999767  recall=0.032018  TP(round)=4290
  top_pct= 5.0%  K= 21,457  precision=0.999953  recall=0.160136  TP(round)=21456
  ✅ внутрішньо узгоджено; implied n_pos_test (з найбільшого K) ≈ 133986.0

----- UNSW-F0 -----
  top_pct= 0.1%  K=    309  precision=1.000000  recall=0.012187  TP(round)=309
  top_pct= 0.5%  K=  1,545  precision=1.000000  recall=0.060937  TP(roun

## 4. Звірка implied test-розмірів з frozen split (Step 5C.2)

`implied n_pos_test`, виведений з Top-K таблиць у розділі 3, звіряється з абсолютними
числами, зафіксованими окремим кроком Step 5C.2 (`step5c2_dedup_and_split_spec.ipynb`),
до будь-якого моделювання: CIC test attack=133,986 (n=429,140, attack_rate=31.2220%),
UNSW test attack=25,354 (n=308,935, attack_rate=8.2069%). Ці числа не є частиною
in-memory стану Step 5D.2 — вони походять з окремого, раніше замороженого кроку.

In [6]:
FROZEN_SPLIT_SIZES = {
    # (n_test, n_pos_test) — зафіксовано в Step 5C.2, ДО Step 5D.2
    "CIC": dict(n_test=429_140, n_pos_test=133_986),
    "UNSW": dict(n_test=308_935, n_pos_test=25_354),
}

print("Звірка implied n_pos_test (з Top-K, розділ 3) проти frozen Step 5C.2 split:")
for dataset_name in ["CIC", "UNSW"]:
    frozen = FROZEN_SPLIT_SIZES[dataset_name]
    for feature_set_name in ["F0", "F1"]:
        implied = implied_n_pos_by_model[(dataset_name, feature_set_name)]
        if implied is None:
            print(f"  ⚠ {dataset_name}-{feature_set_name}: implied n_pos невизначений, пропущено")
            continue
        diff = abs(implied - frozen["n_pos_test"])
        status = "✅" if diff <= 1.0 else "❌"
        print(
            f"  {status} {dataset_name}-{feature_set_name}: implied n_pos_test≈{implied:.1f} "
            f"vs frozen n_pos_test={frozen['n_pos_test']} (Δ={diff:.1f})"
        )
        assert diff <= 1.0, (
            f"{dataset_name}-{feature_set_name}: Top-K таблиця неявно припускає інший розмір "
            f"позитивного класу test, ніж зафіксовано у Step 5C.2 -- можлива розсинхронізація "
            f"split між кроками 5C.2 і 5D.2"
        )

    # також перевіряємо K для top_pct=100%-подібного максимального рядка проти n_test,
    # якщо в збереженій таблиці є достатньо великий top_pct
    for feature_set_name in ["F0", "F1"]:
        df = topk_saved[(dataset_name, feature_set_name)]
        row_max = df.loc[df["K"].idxmax()]
        frac = float(row_max["top_pct"].rstrip("%")) / 100.0
        implied_n_total = row_max["K"] / frac
        diff_n = abs(implied_n_total - frozen["n_test"])
        status_n = "✅" if diff_n <= 2.0 else "⚠"
        print(
            f"  {status_n} {dataset_name}-{feature_set_name}: implied n_test з K/top_pct "
            f"≈{implied_n_total:.1f} vs frozen n_test={frozen['n_test']} (Δ={diff_n:.1f})"
        )


Звірка implied n_pos_test (з Top-K, розділ 3) проти frozen Step 5C.2 split:
  ✅ CIC-F0: implied n_pos_test≈133986.0 vs frozen n_pos_test=133986 (Δ=0.0)
  ✅ CIC-F1: implied n_pos_test≈133986.0 vs frozen n_pos_test=133986 (Δ=0.0)
  ✅ CIC-F0: implied n_test з K/top_pct ≈429140.0 vs frozen n_test=429140 (Δ=0.0)
  ✅ CIC-F1: implied n_test з K/top_pct ≈429140.0 vs frozen n_test=429140 (Δ=0.0)
  ✅ UNSW-F0: implied n_pos_test≈25354.0 vs frozen n_pos_test=25354 (Δ=0.0)
  ✅ UNSW-F1: implied n_pos_test≈25354.0 vs frozen n_pos_test=25354 (Δ=0.0)
  ⚠ UNSW-F0: implied n_test з K/top_pct ≈308940.0 vs frozen n_test=308935 (Δ=5.0)
  ⚠ UNSW-F1: implied n_test з K/top_pct ≈308940.0 vs frozen n_test=308935 (Δ=5.0)


## 5. Незалежне ВІДТВОРЕННЯ frozen-протоколу (не ретюнінг) — свіже завантаження і повний перезапуск

Це не in-memory перевикористання Step 5D.2. Дані завантажуються з диска заново, split і
feature contract будуються заново (тим самим кодом, що й Step 5D.1/5C.2, скопійованим сюди
як самодостатній зафіксований протокол), модель навчається з ідентичним seed=42 і без
жодного пошуку гіперпараметрів. Якщо ця незалежна reproduction збігається з `results_saved`
у межах допуску — це і є справжня незалежна перевірка, а не самоперевірка коду самим собою.


In [7]:
import lightgbm as lgb
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_score, recall_score, f1_score,
)

RANDOM_STATE = 42

LGB_PARAMS = dict(
    objective="binary", metric="None", seed=RANDOM_STATE,
    num_leaves=15, learning_rate=0.05, min_data_in_leaf=50,
)
NUM_BOOST_ROUND = 500
EARLY_STOPPING_ROUNDS = 30
TOP_K_FRACTIONS = [0.001, 0.005, 0.01, 0.05]

FROZEN_BOUNDARIES = {
    "CIC": dict(
        file="cic_ids2017_with_cve_context.parquet", time_col="Timestamp",
        label_col="attack_label", benign_value="BENIGN",
        boundary_1=pd.Timestamp("2017-07-06 09:28:00"),
        boundary_2=pd.Timestamp("2017-07-07 03:23:00"),
    ),
    "UNSW": dict(
        file="unsw_nb15_with_cve_context_dedup.parquet", time_col="Stime",
        label_col="attack_cat", benign_value="Normal",
        boundary_1=pd.Timestamp("2015-02-18 05:36:51"),
        boundary_2=pd.Timestamp("2015-02-18 08:58:36"),
    ),
}

def unweighted_average_precision(preds, train_data):
    y_true = train_data.get_label()
    return "ap", average_precision_score(y_true, preds), True

print("Frozen params (незмінні відносно Step 5D.2):")
print(LGB_PARAMS, "num_boost_round=", NUM_BOOST_ROUND, "early_stopping_rounds=", EARLY_STOPPING_ROUNDS)


Frozen params (незмінні відносно Step 5D.2):
{'objective': 'binary', 'metric': 'None', 'seed': 42, 'num_leaves': 15, 'learning_rate': 0.05, 'min_data_in_leaf': 50} num_boost_round= 500 early_stopping_rounds= 30


In [8]:
# ---------- 5.1 Свіже завантаження з диска ----------
raw = {}
for name, cfg in FROZEN_BOUNDARIES.items():
    df = pd.read_parquet(ENRICHED_DIR / cfg["file"])
    raw[name] = df
    print(f"{name}: свіжо завантажено {len(df):,} рядків, {len(df.columns)} колонок з {cfg['file']}")


CIC: свіжо завантажено 2,827,677 рядків, 96 колонок з cic_ids2017_with_cve_context.parquet
UNSW: свіжо завантажено 2,059,418 рядків, 62 колонок з unsw_nb15_with_cve_context_dedup.parquet


In [9]:
# ---------- 5.2 Feature-contract F0: FROZEN EXPLICIT ORDERED LISTS (не classify_column()) ----------
# forensic-висновок Step 5D.2e/5D.2f: LightGBM у цьому середовищі чутливий до ПОРЯДКУ колонок,
# навіть при ідентичному наборі ознак/даних/seed/гіперпараметрах. Тому F0 більше НЕ
# реконструюється алгоритмічно через classify_column() (це давало інший порядок при тому
# самому наборі і саме це було коренем розбіжності UNSW 33->277 / 290->249). F0 тепер
# буквально скопійований з DATASET_CONFIG[name]["f0_features"] оригінального
# step5d2_f0_f1_baseline.ipynb -- порядок є частиною frozen model contract, не лише набір.
FROZEN_F0_FEATURES = {
    "CIC": [
        "Flow Duration", "Total Fwd Packets", "Total Backward Packets", "Total Length of Fwd Packets",
        "Total Length of Bwd Packets", "Fwd Packet Length Max", "Fwd Packet Length Min",
        "Fwd Packet Length Mean", "Fwd Packet Length Std", "Bwd Packet Length Max",
        "Bwd Packet Length Min", "Bwd Packet Length Mean", "Bwd Packet Length Std",
        "Flow Bytes/s", "Flow Packets/s", "Flow IAT Mean", "Flow IAT Std", "Flow IAT Max",
        "Flow IAT Min", "Fwd IAT Total", "Fwd IAT Mean", "Fwd IAT Std", "Fwd IAT Max",
        "Fwd IAT Min", "Bwd IAT Total", "Bwd IAT Mean", "Bwd IAT Std", "Bwd IAT Max",
        "Bwd IAT Min", "Fwd PSH Flags", "Bwd PSH Flags", "Fwd URG Flags", "Bwd URG Flags",
        "Fwd Header Length", "Bwd Header Length", "Fwd Packets/s", "Bwd Packets/s",
        "Min Packet Length", "Max Packet Length", "Packet Length Mean", "Packet Length Std",
        "Packet Length Variance", "FIN Flag Count", "SYN Flag Count", "RST Flag Count",
        "PSH Flag Count", "ACK Flag Count", "URG Flag Count", "CWE Flag Count", "ECE Flag Count",
        "Down/Up Ratio", "Average Packet Size", "Avg Fwd Segment Size", "Avg Bwd Segment Size",
        "Fwd Avg Bytes/Bulk", "Fwd Avg Packets/Bulk", "Fwd Avg Bulk Rate", "Bwd Avg Bytes/Bulk",
        "Bwd Avg Packets/Bulk", "Bwd Avg Bulk Rate", "Subflow Fwd Packets", "Subflow Fwd Bytes",
        "Subflow Bwd Packets", "Subflow Bwd Bytes", "Init_Win_bytes_forward",
        "Init_Win_bytes_backward", "act_data_pkt_fwd", "min_seg_size_forward", "Active Mean",
        "Active Std", "Active Max", "Active Min", "Idle Mean", "Idle Std", "Idle Max",
        "Idle Min", "protocol",
    ],
    "UNSW": [
        "state", "dur", "sbytes", "dbytes", "sttl", "dttl", "sloss", "dloss", "Sload", "Dload",
        "Spkts", "Dpkts", "swin", "dwin", "stcpb", "dtcpb", "smeansz", "dmeansz", "trans_depth",
        "res_bdy_len", "Sjit", "Djit", "Sintpkt", "Dintpkt", "tcprtt", "synack", "ackdat",
        "is_sm_ips_ports", "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login", "ct_ftp_cmd",
        "ct_srv_src", "ct_srv_dst", "ct_dst_ltm", "ct_src_ltm", "ct_src_dport_ltm",
        "ct_dst_sport_ltm", "ct_dst_src_ltm", "proto", "service",
    ],
}

f0_reproduced = {}
for dataset_key in ["CIC", "UNSW"]:
    df = raw[dataset_key]
    f0_cols = list(FROZEN_F0_FEATURES[dataset_key])
    missing = [c for c in f0_cols if c not in df.columns]
    assert not missing, f"{dataset_key}: frozen F0 посилається на відсутні колонки: {missing}"
    f0_reproduced[dataset_key] = f0_cols
    print(f"{dataset_key}: F0 = {len(f0_cols)} predictors (frozen explicit order, literal copy з оригіналу)")

EXPECTED_F0_COUNTS = {"CIC": 77, "UNSW": 41}
for dataset_key, expected in EXPECTED_F0_COUNTS.items():
    n = len(f0_reproduced[dataset_key])
    assert n == expected, (
        f"{dataset_key}: frozen F0 має {n} ознак, очікувалось {expected} (frozen Step 5D.1) "
        f"-- контракт розійшовся!"
    )
    print(f"✅ {dataset_key}: F0={n} == frozen Step 5D.1 F0={expected}")


CIC: F0 = 77 predictors (frozen explicit order, literal copy з оригіналу)
UNSW: F0 = 41 predictors (frozen explicit order, literal copy з оригіналу)
✅ CIC: F0=77 == frozen Step 5D.1 F0=77
✅ UNSW: F0=41 == frozen Step 5D.1 F0=41


In [10]:
# ---------- 5.2b Діагностична cross-check: classify_column() як НЕЗАЛЕЖНА перевірка НАБОРУ ----------
# Це вже НЕ джерело F0 (це і був корінь розбіжності Step 5D.2b/UNSW -- classify_column()
# давав інший ПОРЯДОК при тому самому наборі). Залишено лише як sanity-check: чи роль-
# класифікація raw.columns і надалі узгоджується МНОЖИНОЮ з frozen F0 -- виявляє дрейф
# схеми вихідного parquet (нову/зниклу колонку), а не використовується для побудови X.
import re

IP_PATTERN = re.compile(r"^(srcip|src_ip|source_ip|dstip|dst_ip|destination_ip|src_addr|dst_addr|srcaddr|dstaddr)$", re.IGNORECASE)
PORT_PATTERN = re.compile(r"^(sport|src_port|source_port|dsport|dst_port|destination_port)$", re.IGNORECASE)
PROTO_PATTERN = re.compile(r"^(proto|protocol)$", re.IGNORECASE)
FLOWID_PATTERN = re.compile(r"(flow.?id|^id$)", re.IGNORECASE)
PROVENANCE_PATTERN = re.compile(r"^(__source_file|__.*)$", re.IGNORECASE)

TIME_COLS = {
    "CIC": {"Timestamp"},
    "UNSW": {"Stime", "Ltime"},
}
LABEL_COL = {"CIC": "attack_label", "UNSW": "attack_cat"}

def classify_column(col, dataset_key, df):
    if col.startswith("cve_ctx_"):
        return "CTI (cve_ctx_*)"
    if col == LABEL_COL[dataset_key]:
        return "target"
    if col in TIME_COLS[dataset_key]:
        return "time"
    if IP_PATTERN.match(col):
        return "network identifier (IP)"
    if PORT_PATTERN.match(col):
        return "network categorical feature (port)"
    if FLOWID_PATTERN.search(col):
        return "identifier (flow ID)"
    if PROVENANCE_PATTERN.match(col):
        return "provenance metadata"
    if col.lower() == "label":
        return "target-derived (redundant)"
    if PROTO_PATTERN.match(col):
        return "predictor (categorical)"
    if col.lower() == "service" and "cve_ctx_service" in df.columns:
        return "predictor (overlap risk)"
    return "predictor"

INCLUDE_ROLES = {"predictor", "predictor (categorical)", "predictor (overlap risk)"}

for dataset_key in ["CIC", "UNSW"]:
    df = raw[dataset_key]
    roles = {col: classify_column(col, dataset_key, df) for col in df.columns}
    f0_via_classify = set(c for c, r in roles.items() if r in INCLUDE_ROLES)
    f0_frozen_set = set(f0_reproduced[dataset_key])
    if f0_via_classify == f0_frozen_set:
        print(f"✅ {dataset_key}: множина classify_column()-ознак == множина frozen F0 "
              f"(SET-перевірка, НЕ порядок -- порядок береться виключно з frozen списку)")
    else:
        only_classify = f0_via_classify - f0_frozen_set
        only_frozen = f0_frozen_set - f0_via_classify
        print(f"⚠️ {dataset_key}: РОЗБІЖНІСТЬ множин! лише в classify_column(): {only_classify}  "
              f"лише у frozen: {only_frozen} -- можливий дрейф схеми вихідного parquet, "
              f"потребує розбору перед довірою до результатів.")


✅ CIC: множина classify_column()-ознак == множина frozen F0 (SET-перевірка, НЕ порядок -- порядок береться виключно з frozen списку)
✅ UNSW: множина classify_column()-ознак == множина frozen F0 (SET-перевірка, НЕ порядок -- порядок береться виключно з frozen списку)


In [11]:
# ---------- 5.3 F1 = F0 + frozen CTI-додавання (буквально ті самі списки, що заморожені у Step 5C.2/5D.2) ----------
F1_ADDITIONS = {
    "CIC": [
        "cve_ctx_service", "cve_ctx_pool_size", "cve_ctx_n_epss", "cve_ctx_epss_coverage",
        "cve_ctx_service_max_epss", "cve_ctx_n_generic_os_matches", "cve_ctx_n_specific_product_matches",
    ],
    "UNSW": [
        "cve_ctx_service", "cve_ctx_pool_size", "cve_ctx_epss_coverage",
        "cve_ctx_n_generic_os_matches", "cve_ctx_n_specific_product_matches",
    ],
}

FEATURE_SETS = {}
for dataset_key in ["CIC", "UNSW"]:
    f0 = f0_reproduced[dataset_key]
    f1 = f0 + F1_ADDITIONS[dataset_key]
    for c in F1_ADDITIONS[dataset_key]:
        assert c in raw[dataset_key].columns, f"{dataset_key}: {c} відсутня у свіжо завантаженому parquet"
    FEATURE_SETS[(dataset_key, "F0")] = f0
    FEATURE_SETS[(dataset_key, "F1")] = f1
    print(f"{dataset_key}-F0={len(f0)}  {dataset_key}-F1={len(f1)} (+{len(F1_ADDITIONS[dataset_key])} CTI)")

EXPECTED_FEATURE_COUNTS = {("CIC", "F0"): 77, ("CIC", "F1"): 84, ("UNSW", "F0"): 41, ("UNSW", "F1"): 46}
for key, expected in EXPECTED_FEATURE_COUNTS.items():
    n = len(FEATURE_SETS[key])
    assert n == expected, f"{key}: перевідтворена кількість ознак={n}, очікувалось {expected}"
print("✅ Усі 4 feature-set перевідтворені з очікуваною кількістю ознак (77/84/41/46)")


CIC-F0=77  CIC-F1=84 (+7 CTI)
UNSW-F0=41  UNSW-F1=46 (+5 CTI)
✅ Усі 4 feature-set перевідтворені з очікуваною кількістю ознак (77/84/41/46)


In [12]:
# ---------- 5.4 Target, ct_ftp_cmd numeric fix, temporal split (frozen межі), categorical vocab fit-on-train ----------
splits = {}
for dataset_key, cfg in FROZEN_BOUNDARIES.items():
    df = raw[dataset_key].copy()
    df["_is_attack"] = (df[cfg["label_col"]] != cfg["benign_value"]).astype(int)

    if dataset_key == "UNSW" and "ct_ftp_cmd" in df.columns:
        # той самий, окремо замороженний Step 5D.1 audit-вердикт: ' ' -> NaN, numeric count
        df["ct_ftp_cmd"] = pd.to_numeric(df["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")

    t = pd.to_datetime(df[cfg["time_col"]], unit="s" if dataset_key == "UNSW" else None, errors="raise") \
        if dataset_key == "UNSW" and pd.api.types.is_integer_dtype(df[cfg["time_col"]]) \
        else pd.to_datetime(df[cfg["time_col"]])

    train_mask = t < cfg["boundary_1"]
    val_mask = (t >= cfg["boundary_1"]) & (t < cfg["boundary_2"])
    test_mask = t >= cfg["boundary_2"]

    train_df, val_df, test_df = df[train_mask].copy(), df[val_mask].copy(), df[test_mask].copy()
    train_t, val_t, test_t = t[train_mask], t[val_mask], t[test_mask]

    assert train_t.max() < val_t.min(), f"{dataset_key}: train/val temporal overlap"
    assert val_t.max() < test_t.min(), f"{dataset_key}: val/test temporal overlap"

    n_test, n_pos_test = len(test_df), int(test_df["_is_attack"].sum())
    frozen = FROZEN_SPLIT_SIZES[dataset_key]
    assert n_test == frozen["n_test"], (
        f"{dataset_key}: перевідтворений n_test={n_test:,} != frozen n_test={frozen['n_test']:,}"
    )
    assert n_pos_test == frozen["n_pos_test"], (
        f"{dataset_key}: перевідтворений n_pos_test={n_pos_test:,} != frozen n_pos_test={frozen['n_pos_test']:,}"
    )
    print(f"✅ {dataset_key}: перевідтворений test split точно збігається з frozen Step 5C.2 "
          f"(n_test={n_test:,}, n_pos_test={n_pos_test:,})")

    categorical_candidates = [c for c in FEATURE_SETS[(dataset_key, "F1")]
                              if not pd.api.types.is_numeric_dtype(df[c]) or df[c].dtype == bool]
    if dataset_key == "UNSW":
        # ct_ftp_cmd вже numeric після конвертації вище; проте лишаємо явну перевірку,
        # що вона НЕ потрапила у categorical_candidates
        assert "ct_ftp_cmd" not in categorical_candidates
    for col in categorical_candidates:
        train_categories = pd.Categorical(train_df[col]).categories
        train_df[col] = pd.Categorical(train_df[col], categories=train_categories)
        val_df[col] = pd.Categorical(val_df[col], categories=train_categories)
        test_df[col] = pd.Categorical(test_df[col], categories=train_categories)

    splits[dataset_key] = dict(train=train_df, val=val_df, test=test_df, categorical_candidates=categorical_candidates)

print("\nКатегоріальні колонки, перевідтворені для кожного датасету:")
for k, v in splits.items():
    print(f"  {k}: {v['categorical_candidates']}")


✅ CIC: перевідтворений test split точно збігається з frozen Step 5C.2 (n_test=429,140, n_pos_test=133,986)
✅ UNSW: перевідтворений test split точно збігається з frozen Step 5C.2 (n_test=308,935, n_pos_test=25,354)

Категоріальні колонки, перевідтворені для кожного датасету:
  CIC: ['protocol', 'cve_ctx_service']
  UNSW: ['state', 'proto', 'service', 'cve_ctx_service']


## 6. Незалежне повторне навчання frozen LightGBM-протоколу (той самий seed, без пошуку)

Кожна з 4 моделей навчається один раз, з тим самим зафіксованим значенням `seed=42` та
frozen гіперпараметрами: `objective=binary`, `num_leaves=15`, `learning_rate=0.05`,
`min_data_in_leaf=50`, `num_boost_round=500`, `early_stopping_rounds=30`,
`feval=unweighted_average_precision`, `valid_sets=[val]` (test ніколи не передається в
`lgb.train`). Це відтворення, а не новий пошук — жодне значення тут не підбирається за
результатом. (Як показано у розділі 7a: `seed=42` сам по собі НЕ гарантує побітового
детермінізму LightGBM без додаткових прапорців -- тому формулювання тут навмисно не називає
це "ідентичним протоколом".)

In [13]:
def train_and_evaluate_reproduction(dataset_key, feature_set_name, param_overrides=None):
    features = FEATURE_SETS[(dataset_key, feature_set_name)]
    train_df, val_df, test_df = splits[dataset_key]["train"], splits[dataset_key]["val"], splits[dataset_key]["test"]

    X_train, y_train = train_df[features], train_df["_is_attack"]
    X_val, y_val = val_df[features], val_df["_is_attack"]
    X_test, y_test = test_df[features], test_df["_is_attack"]

    # LIST equality, НЕ set equality (Step 5D.2e/5D.2f forensic: set-рівність недостатня --
    # LightGBM чутливий до порядку колонок навіть при ідентичному наборі ознак).
    assert list(X_train.columns) == features, f"{dataset_key}-{feature_set_name}: X_train порядок колонок розійшовся з frozen features!"
    assert list(X_val.columns) == features, f"{dataset_key}-{feature_set_name}: X_val порядок колонок розійшовся з frozen features!"
    assert list(X_test.columns) == features, f"{dataset_key}-{feature_set_name}: X_test порядок колонок розійшовся з frozen features!"

    cat_cols = [c for c in features if X_train[c].dtype.name == "category"]

    n_pos_train = int(y_train.sum())
    n_neg_train = len(y_train) - n_pos_train
    w = n_neg_train / n_pos_train
    params = dict(LGB_PARAMS, scale_pos_weight=w)
    if param_overrides:
        # ВИКОРИСТОВУЄТЬСЯ ЛИШЕ forensic-діагностикою (розділ 7d) -- НЕ frozen-протоколом.
        # За замовчуванням (param_overrides=None) функція поводиться ІДЕНТИЧНО до frozen
        # версії, що й раніше -- розділ 6 викликає її без цього аргументу.
        params.update(param_overrides)

    lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols if cat_cols else "auto")
    lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train, categorical_feature=cat_cols if cat_cols else "auto")

    model = lgb.train(
        params, lgb_train, num_boost_round=NUM_BOOST_ROUND,
        valid_sets=[lgb_val], valid_names=["val"],
        feval=unweighted_average_precision,
        callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS, verbose=False),
                  lgb.log_evaluation(period=0)],
    )
    best_iter = model.best_iteration
    val_scores = model.predict(X_val, num_iteration=best_iter)
    test_scores = model.predict(X_test, num_iteration=best_iter)

    def topk_metrics(y_true, y_score):
        rows = []
        order = np.argsort(-y_score, kind="mergesort")
        y_true_arr = y_true.values
        for frac in TOP_K_FRACTIONS:
            k = max(1, int(round(len(y_true_arr) * frac)))
            top_idx = order[:k]
            tp = y_true_arr[top_idx].sum()
            rows.append(dict(top_pct=f"{frac:.1%}", K=k,
                             precision=tp / k, recall=tp / y_true_arr.sum() if y_true_arr.sum() else np.nan))
        return pd.DataFrame(rows)

    y_pred_test = (test_scores >= 0.5).astype(int)
    result = dict(
        dataset=dataset_key, feature_set=feature_set_name, n_features=len(features),
        best_iteration=best_iter, scale_pos_weight=w,
        val_ap=average_precision_score(y_val, val_scores),
        val_roc_auc=roc_auc_score(y_val, val_scores),
        test_ap=average_precision_score(y_test, test_scores),
        test_roc_auc=roc_auc_score(y_test, test_scores),
        test_precision=precision_score(y_test, y_pred_test, zero_division=0),
        test_recall=recall_score(y_test, y_pred_test, zero_division=0),
        test_f1=f1_score(y_test, y_pred_test, zero_division=0),
    )
    topk_df = topk_metrics(y_test, test_scores)
    return result, topk_df, test_scores, y_test.values, test_df.index.values


reproduced_results, reproduced_topk, raw_test_scores = [], {}, {}
for dataset_key in ["CIC", "UNSW"]:
    for feature_set_name in ["F0", "F1"]:
        print(f"\n{'='*20} {dataset_key}-{feature_set_name} (незалежне відтворення) {'='*20}")
        result, topk_df, test_scores, y_test_vals, test_index = train_and_evaluate_reproduction(dataset_key, feature_set_name)
        print(f"best_iteration={result['best_iteration']}  test_AP={result['test_ap']:.6f}  "
              f"test_ROC-AUC={result['test_roc_auc']:.4f}")
        reproduced_results.append(result)
        reproduced_topk[(dataset_key, feature_set_name)] = topk_df
        raw_test_scores[(dataset_key, feature_set_name)] = dict(
            scores=test_scores, y_true=y_test_vals, index=test_index
        )

reproduced_df = pd.DataFrame(reproduced_results).set_index(["dataset", "feature_set"])
display(reproduced_df[["n_features", "best_iteration", "val_ap", "val_roc_auc",
                       "test_ap", "test_roc_auc", "test_precision", "test_recall", "test_f1"]])



==================== CIC-F0 (незалежне відтворення) ====================
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 265848, number of negative: 1713498
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.226174 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14089
[LightGBM] [Info] Number of data points in the train set: 1979346, number of used features: 69
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.134311 -> initscore=-1.863367
[LightGBM] [Info] Start training from score -1.863367
best_iteration=4  test_AP=0.710369  test_ROC-AUC=0.7406

==================== CIC-F1 (незалежне відтворення) ====================
[LightGBM] [Warning] Found whitespace in feature_names, replace with und

n_features  best_iteration    val_ap  val_roc_auc  \
dataset feature_set                                                      
CIC     F0                   77               4  0.778550     0.823532   
        F1                   84             118  0.676896     0.765135   
UNSW    F0                   41              33  0.964633     0.997017   
        F1                   46             290  0.979776     0.997882   

                      test_ap  test_roc_auc  test_precision  test_recall  \
dataset feature_set                                                        
CIC     F0           0.710369      0.740553        0.000000     0.000000   
        F1           0.974028      0.982162        0.997403     0.604780   
UNSW    F0           0.973652      0.997671        0.778539     0.999842   
        F1           0.983969      0.998247        0.779184     0.998028   

                      test_f1  
dataset feature_set            
CIC     F0           0.000000  
        F1           0.752984  
UNSW    F0           0.875421  
        F1           0.875132

## 6b. Незалежний ΔAP напряму з reproduced_df (не через results_saved)

Замикає весь ланцюг незалежності: saved results → свіже завантаження → незалежна побудова
feature contract → незалежний split → незалежне навчання → незалежні прогнози → незалежний
AP → **незалежний ΔAP**, порахований тут з нуля з `reproduced_df`, а не переписаний з
`results_saved`/`delta_saved`.

In [14]:
print("Незалежний ΔAP, порахований напряму з reproduced_df (розділ 6):")
independent_delta_rows = []
for dataset_key in ["CIC", "UNSW"]:
    ap_f0_rep = reproduced_df.loc[(dataset_key, "F0"), "test_ap"]
    ap_f1_rep = reproduced_df.loc[(dataset_key, "F1"), "test_ap"]
    delta_rep = ap_f1_rep - ap_f0_rep
    independent_delta_rows.append(dict(dataset=dataset_key, F0_AP=ap_f0_rep, F1_AP=ap_f1_rep, delta_AP=delta_rep))
    print(f"  {dataset_key}: F0_AP={ap_f0_rep:.6f}  F1_AP={ap_f1_rep:.6f}  ΔAP={delta_rep:+.6f}")

independent_delta_df = pd.DataFrame(independent_delta_rows).set_index("dataset")

DELTA_AP_TOL = 1e-4  # той самий допуск, що й у розділі 7 для AP/ROC-AUC/Precision/Recall/F1

print("\nЗвірка незалежного ΔAP (з reproduced_df) проти збереженого delta_saved:")
delta_ap_match = {}
for dataset_key in ["CIC", "UNSW"]:
    a = float(independent_delta_df.loc[dataset_key, "delta_AP"])
    b = float(delta_saved_idx.loc[dataset_key, "delta_AP"])
    diff = abs(a - b)
    ok = diff <= DELTA_AP_TOL
    delta_ap_match[dataset_key] = ok
    print(f"  {'✅' if ok else '❌'} {dataset_key}: independent ΔAP={a:+.6f}  saved ΔAP={b:+.6f}  Δ={diff:.2e}")

if not all(delta_ap_match.values()):
    print(
        "\n⚠ Розбіжність ΔAP для датасету(ів), позначених ❌ вище, НЕ зупиняє виконання "
        "notebook -- це навмисне рішення: справжня розбіжність результатів навчання є "
        "знахідкою аудиту, яку треба задокументувати й дослідити нижче (розділи 7-7d), а не "
        "приховати аварійною зупинкою kernel."
    )


Незалежний ΔAP, порахований напряму з reproduced_df (розділ 6):
  CIC: F0_AP=0.710369  F1_AP=0.974028  ΔAP=+0.263659
  UNSW: F0_AP=0.973652  F1_AP=0.983969  ΔAP=+0.010317

Звірка незалежного ΔAP (з reproduced_df) проти збереженого delta_saved:
  ✅ CIC: independent ΔAP=+0.263659  saved ΔAP=+0.263659  Δ=0.00e+00
  ✅ UNSW: independent ΔAP=+0.010317  saved ΔAP=+0.010317  Δ=3.82e-17


## 7. Звірка незалежного відтворення проти збережених артефактів (з допуском)

Допуск на числову нестабільність (порядок потоків/фіч, версії бібліотек) береться дуже
вузьким: `1e-4` для AP/ROC-AUC/Precision/Recall/F1, точна рівність для `best_iteration`
(детерміноване число при фіксованому seed).

In [15]:
AP_TOL = 1e-4
BEST_ITER_MUST_MATCH_EXACTLY = True

print("=" * 78)
print("Звірка: незалежне відтворення (розділ 6) vs збережені step5d2_f0_f1_results_h_flow.csv")
print("=" * 78)

all_match = True
dataset_all_match = {}
for dataset_key in ["CIC", "UNSW"]:
    this_dataset_match = True
    for feature_set_name in ["F0", "F1"]:
        rep = reproduced_df.loc[(dataset_key, feature_set_name)]
        sav = results_idx.loc[(dataset_key, feature_set_name)]

        rows_to_check = [
            ("test_ap", AP_TOL), ("test_roc_auc", AP_TOL),
            ("test_precision", AP_TOL), ("test_recall", AP_TOL), ("test_f1", AP_TOL),
        ]
        print(f"\n{dataset_key}-{feature_set_name}:")
        for col, tol in rows_to_check:
            diff = abs(float(rep[col]) - float(sav[col]))
            ok = diff <= tol
            all_match &= ok
            this_dataset_match &= ok
            print(f"  {'✅' if ok else '❌'} {col}: reproduced={rep[col]:.6f}  saved={sav[col]:.6f}  Δ={diff:.2e}")

        rep_iter = int(rep["best_iteration"])
        sav_iter = int(sav["best_iteration"])
        iter_ok = rep_iter == sav_iter
        all_match &= iter_ok
        this_dataset_match &= iter_ok
        print(f"  {'✅' if iter_ok else '❌'} best_iteration: reproduced={rep_iter}  saved={sav_iter}")

    dataset_all_match[dataset_key] = this_dataset_match

print("\n" + "=" * 78)
print("Підсумок ПО ДАТАСЕТАХ ОКРЕМО (жодного спільного твердження на обидва датасети відразу):")
print("=" * 78)
for dataset_key in ["CIC", "UNSW"]:
    if dataset_all_match[dataset_key]:
        print(
            f"✅ {dataset_key}: незалежне відтворення ТОЧНО узгоджується зі збереженими "
            f"результатами Step 5D.2 (F0 і F1) у межах tolerance {AP_TOL:.0e} для "
            f"AP/ROC-AUC/Precision/Recall/F1 і ТОЧНО (без допуску) для best_iteration. Це "
            f"підтверджує відтворюваність заявлених результатів {dataset_key} за frozen "
            f"experimental protocol та виключає залежність від in-memory стану оригінального "
            f"прогону.\n"
        )
    else:
        f0_rep, f0_sav = reproduced_df.loc[(dataset_key, "F0")], results_idx.loc[(dataset_key, "F0")]
        f1_rep, f1_sav = reproduced_df.loc[(dataset_key, "F1")], results_idx.loc[(dataset_key, "F1")]
        print(
            f"❌ {dataset_key}: незалежне відтворення НЕ узгоджується повністю зі збереженими "
            f"результатами Step 5D.2 -- "
            f"F0: AP reproduced={f0_rep['test_ap']:.6f} vs saved={f0_sav['test_ap']:.6f}, "
            f"best_iteration {int(f0_rep['best_iteration'])} vs {int(f0_sav['best_iteration'])}; "
            f"F1: AP reproduced={f1_rep['test_ap']:.6f} vs saved={f1_sav['test_ap']:.6f}, "
            f"best_iteration {int(f1_rep['best_iteration'])} vs {int(f1_sav['best_iteration'])}. "
            f"Тому оригінальні результати {dataset_key} Step 5D.2 НАРАЗІ НЕ можна вважати "
            f"незалежно відтвореними, доки не будуть узгоджені execution environment, "
            f"налаштування детермінізму LightGBM та повна конфігурація тренування -- "
            f"див. forensic-діагностику нижче (розділи 7a-7d).\n"
        )


Звірка: незалежне відтворення (розділ 6) vs збережені step5d2_f0_f1_results_h_flow.csv

CIC-F0:
  ✅ test_ap: reproduced=0.710369  saved=0.710369  Δ=0.00e+00
  ✅ test_roc_auc: reproduced=0.740553  saved=0.740553  Δ=0.00e+00
  ✅ test_precision: reproduced=0.000000  saved=0.000000  Δ=0.00e+00
  ✅ test_recall: reproduced=0.000000  saved=0.000000  Δ=0.00e+00
  ✅ test_f1: reproduced=0.000000  saved=0.000000  Δ=0.00e+00
  ✅ best_iteration: reproduced=4  saved=4

CIC-F1:
  ✅ test_ap: reproduced=0.974028  saved=0.974028  Δ=1.11e-16
  ✅ test_roc_auc: reproduced=0.982162  saved=0.982162  Δ=1.11e-16
  ✅ test_precision: reproduced=0.997403  saved=0.997403  Δ=0.00e+00
  ✅ test_recall: reproduced=0.604780  saved=0.604780  Δ=0.00e+00
  ✅ test_f1: reproduced=0.752984  saved=0.752984  Δ=0.00e+00
  ✅ best_iteration: reproduced=118  saved=118

UNSW-F0:
  ✅ test_ap: reproduced=0.973652  saved=0.973652  Δ=0.00e+00
  ✅ test_roc_auc: reproduced=0.997671  saved=0.997671  Δ=1.11e-16
  ✅ test_precision: reproduc

## 7a. Метадані середовища виконання (версії бібліотек, seeds) -- записується завжди

Записується незалежно від результату звірки в розділі 7, як частина audit trail. Оригінальний
Step 5D.2 не зберігав цю інформацію, тому пряме порівняння версій між прогонами наразі
неможливе -- це задокументована прогалина, а не привід уникнути перевірки.

In [16]:
import sys
import sklearn

env_info = dict(
    python=sys.version.split()[0],
    lightgbm=lgb.__version__,
    numpy=np.__version__,
    pandas=pd.__version__,
    sklearn=sklearn.__version__,
)
print("Версії середовища виконання цього аудиту:")
for k, v in env_info.items():
    print(f"  {k}: {v}")
print(f"  lightgbm module path: {lgb.__file__}")

print("\nFrozen LGB_PARAMS (використані у розділі 6, без модифікацій):")
print(LGB_PARAMS)
print(
    "\n⚠ Явно зафіксовано лише seed=42. НЕ зафіксовані окремо: bagging_seed, "
    "feature_fraction_seed, data_random_seed, deterministic, force_col_wise/force_row_wise, "
    "num_threads. seed=42 сам по собі НЕ гарантує побітову відтворюваність побудови дерев "
    "LightGBM між окремими процесами/середовищами виконання -- документація LightGBM не "
    "гарантує детермінізм порядку сумування гістограм при багатопотоковості без явного "
    "deterministic=True."
)


Версії середовища виконання цього аудиту:
  python: 3.11.15
  lightgbm: 4.7.0
  numpy: 2.4.4
  pandas: 3.0.5
  sklearn: 1.9.0
  lightgbm module path: /opt/conda/envs/ids_research/lib/python3.11/site-packages/lightgbm/__init__.py

Frozen LGB_PARAMS (використані у розділі 6, без модифікацій):
{'objective': 'binary', 'metric': 'None', 'seed': 42, 'num_leaves': 15, 'learning_rate': 0.05, 'min_data_in_leaf': 50}

⚠ Явно зафіксовано лише seed=42. НЕ зафіксовані окремо: bagging_seed, feature_fraction_seed, data_random_seed, deterministic, force_col_wise/force_row_wise, num_threads. seed=42 сам по собі НЕ гарантує побітову відтворюваність побудови дерев LightGBM між окремими процесами/середовищами виконання -- документація LightGBM не гарантує детермінізм порядку сумування гістограм при багатопотоковості без явного deterministic=True.


## 7b. Hash-перевірка розділених train/val/test даних і feature contract ОКРЕМО для F0 і F1

Хешується розбита вибірка ОКРЕМО для F0 і F1 (а не лише для F1, підмножиною якого є F0),
щоб explicitly підтвердити, який саме input matrix пішов у навчання F0-моделі, а який --
F1-моделі. Хеші -- справжні SHA-256 fingerprints (від послідовності per-row hash-ів), а не
довільне ціле число. Додатково хешується сам упорядкований список назв ознак
(`feature_names_hash`) і, окремо, ПОВНИЙ feature/type/category contract
(`feature_contract_hash`: назва + dtype кожної ознаки, а для categorical -- ще й точний
список train categories) -- щоб майбутній аудит міг сказати не просто "F0 = 41 ознака", а
"F0 має той самий конкретний ordered feature/type/category contract". Це НЕ доводить збіг з
оригінальним прогоном Step 5D.2 (там такі hash не зберігались -- задокументована прогалина),
але дозволяє детектувати розбіжність вхідних даних між майбутніми повторними запусками цього
самого audit notebook.

In [17]:
import hashlib

def hash_df(df):
    # SHA-256 від послідовності per-row hash-ів (а не int-сума) -- нормальний
    # криптографічний fingerprint для forensic audit trail, а не довільне ціле число.
    row_hashes = pd.util.hash_pandas_object(df, index=True).values
    return hashlib.sha256(row_hashes.tobytes()).hexdigest()

def hash_feature_names(features):
    return hashlib.sha256("\n".join(features).encode()).hexdigest()

def feature_contract_hash(df, features):
    # Фіксує не лише НАЗВИ ознак, а й dtype кожної, а для categorical -- ще й точний
    # список train categories. Це дозволяє відрізнити "та сама кількість ознак" від
    # "той самий конкретний ordered feature/type/category contract".
    parts = []
    for c in features:
        parts.append(f"{c}|{df[c].dtype}")
        if isinstance(df[c].dtype, pd.CategoricalDtype):
            parts.append("|".join(map(str, df[c].cat.categories)))
    return hashlib.sha256("\n".join(parts).encode()).hexdigest()

data_hashes = {}
for dataset_key in ["CIC", "UNSW"]:
    train_df, val_df, test_df = splits[dataset_key]["train"], splits[dataset_key]["val"], splits[dataset_key]["test"]
    label_col = "_is_attack"
    for feature_set_name in ["F0", "F1"]:
        features = FEATURE_SETS[(dataset_key, feature_set_name)]
        key = f"{dataset_key}_{feature_set_name}"
        data_hashes[key] = dict(
            n_features=len(features),
            feature_names_hash=hash_feature_names(features),
            # dtype/category contract хешується на TRAIN (де фіксуються categories),
            # оскільки саме train визначає vocabulary для val/test (fit-on-train).
            feature_contract_hash=feature_contract_hash(train_df, features),
            train_X=hash_df(train_df[features]), train_y=hash_df(train_df[[label_col]]),
            val_X=hash_df(val_df[features]), val_y=hash_df(val_df[[label_col]]),
            test_X=hash_df(test_df[features]), test_y=hash_df(test_df[[label_col]]),
            train_index=hash_df(pd.DataFrame(index=train_df.index)),
            val_index=hash_df(pd.DataFrame(index=val_df.index)),
            test_index=hash_df(pd.DataFrame(index=test_df.index)),
        )
        h = data_hashes[key]
        print(f"{key}: n_features={h['n_features']}  "
              f"feature_contract_hash={h['feature_contract_hash'][:12]}...  "
              f"hash(train_X)={h['train_X'][:12]}...  hash(val_X)={h['val_X'][:12]}...  "
              f"hash(test_X)={h['test_X'][:12]}...")


CIC_F0: n_features=77  feature_contract_hash=ad4d87c800cb...  hash(train_X)=941476a1290f...  hash(val_X)=d4eb14fb6781...  hash(test_X)=290e877a6b64...
CIC_F1: n_features=84  feature_contract_hash=dc883f40b880...  hash(train_X)=62185f3d49a7...  hash(val_X)=f5f741f30454...  hash(test_X)=f726de280b04...
UNSW_F0: n_features=41  feature_contract_hash=4bec05628de7...  hash(train_X)=1d85dbd2cd9c...  hash(val_X)=f44a4cf764f6...  hash(test_X)=20ceee4cef45...
UNSW_F1: n_features=46  feature_contract_hash=4a9804633d15...  hash(train_X)=de1b5f1ae8bc...  hash(val_X)=145ed9cea661...  hash(test_X)=1cfc68b9b8d8...


## 7c. Repeat-stability тест: чи стабільне ПОТОЧНЕ середовище саме по собі

Виконується ТІЛЬКИ для датасетів, де розділ 7 виявив розбіжність (щоб не витрачати час на
датасет, який уже відтворився точно). Той самий frozen `train_and_evaluate_reproduction()`
запускається `N_REPEATS` разів поспіль у цьому самому kernel-процесі з тими самими вхідними
даними.

Якщо всі повтори дають ІДЕНТИЧНИЙ `best_iteration`/`test_ap` -- це доводить лише, що поточний
execution path є стабільним САМ ПО СОБІ. Розбіжність зі збереженими результатами Step 5D.2 тоді
НЕ пояснюється випадковою нестабільністю цього конкретного прогону; її причина може бути
пов'язана з відмінностями execution environment, конфігурації (версія LightGBM, hardware/
threading, невідомий training parameter, реалізація preprocessing) або самого training
execution path оригінального прогону -- а не обов'язково саме "середовищем" у вузькому сенсі.

Якщо ж повтори дають РІЗНІ результати -- це пряма ознака недетермінізму в поточному LightGBM
execution path, і причина розбіжності може лежати вже в межах поточного середовища, а не між
середовищами.

In [18]:
N_REPEATS = 5
repeat_stability = {}

datasets_needing_diagnosis = [d for d in ["CIC", "UNSW"] if not dataset_all_match[d]]
if not datasets_needing_diagnosis:
    print("✅ Усі датасети точно узгоджені у розділі 7 -- repeat-stability діагностика не потрібна.")
else:
    print(f"Датасети, що потребують діагностики: {datasets_needing_diagnosis}")
    for dataset_key in datasets_needing_diagnosis:
        for feature_set_name in ["F0", "F1"]:
            print(f"\n{dataset_key}-{feature_set_name}: {N_REPEATS} повторних запусків "
                  f"train_and_evaluate_reproduction() у цьому ж kernel-процесі (той самий "
                  f"frozen LGB_PARAMS, без модифікацій)...")
            runs = []
            for i in range(N_REPEATS):
                result, *_ = train_and_evaluate_reproduction(dataset_key, feature_set_name)
                runs.append((result["best_iteration"], result["test_ap"]))
                bi, ap = result["best_iteration"], result["test_ap"]
                print(f"    run {i}: best_iteration={bi}  test_ap={ap:.6f}")
            repeat_stability[(dataset_key, feature_set_name)] = runs
            best_iters = {r[0] for r in runs}
            stable = len(best_iters) == 1
            print(f"  {'✅ СТАБІЛЬНО' if stable else '❌ НЕСТАБІЛЬНО'} у межах поточного "
                  f"середовища: {len(best_iters)} унікальних значень best_iteration серед "
                  f"{N_REPEATS} запусків ({sorted(best_iters)}).")
            if stable:
                print(
                    "     Це означає лише, що ПОТОЧНИЙ execution path стабільний сам по собі. "
                    "Розбіжність зі збереженим Step 5D.2 НЕ пояснюється випадковою "
                    "нестабільністю цього прогону; причина може бути в execution environment, "
                    "конфігурації або training execution path оригінального прогону."
                )
            else:
                print(
                    "     Це пряма ознака недетермінізму в ПОТОЧНОМУ LightGBM execution path -- "
                    "причина розбіжності може лежати вже в межах поточного середовища."
                )


✅ Усі датасети точно узгоджені у розділі 7 -- repeat-stability діагностика не потрібна.


## 7d. Діагностичний (НЕ frozen) прогін з явним deterministic-режимом LightGBM

Це ЕКСПЕРИМЕНТАЛЬНА конфігурація, яка НЕ є частиною замороженого протоколу Step 5D.2 і
використовується ТІЛЬКИ для діагностики джерела недетермінізму: `deterministic=True`,
`force_row_wise=True` та явно зафіксовані `bagging_seed`/`feature_fraction_seed`/
`data_random_seed`. Стабільний результат тут НЕ можна інтерпретувати як "ми точно відтворили
оригінальний Step 5D.2" -- лише як перевірку того, чи стабілізує САМЕ ЦЕ налаштування повторні
прогони в поточному середовищі.

In [19]:
DETERMINISTIC_PARAM_OVERRIDES = dict(
    bagging_seed=RANDOM_STATE, feature_fraction_seed=RANDOM_STATE, data_random_seed=RANDOM_STATE,
    deterministic=True, force_row_wise=True,
)
N_REPEATS_DETERMINISTIC = 3
deterministic_stability = {}

if not datasets_needing_diagnosis:
    print("Пропущено -- усі датасети вже точно узгоджені (розділ 7).")
else:
    print("⚠ ЕКСПЕРИМЕНТАЛЬНА конфігурація (НЕ frozen-протокол):", DETERMINISTIC_PARAM_OVERRIDES)
    for dataset_key in datasets_needing_diagnosis:
        for feature_set_name in ["F0", "F1"]:
            print(f"\n{dataset_key}-{feature_set_name}: {N_REPEATS_DETERMINISTIC} прогони з "
                  f"deterministic=True...")
            runs = []
            for i in range(N_REPEATS_DETERMINISTIC):
                result, *_ = train_and_evaluate_reproduction(
                    dataset_key, feature_set_name, param_overrides=DETERMINISTIC_PARAM_OVERRIDES
                )
                runs.append((result["best_iteration"], result["test_ap"]))
                bi, ap = result["best_iteration"], result["test_ap"]
                print(f"    run {i}: best_iteration={bi}  test_ap={ap:.6f}")
            deterministic_stability[(dataset_key, feature_set_name)] = runs
            best_iters = {r[0] for r in runs}
            stable = len(best_iters) == 1
            print(f"  {'✅ СТАБІЛЬНО' if stable else '❌ НЕСТАБІЛЬНО'} з deterministic=True: "
                  f"{len(best_iters)} унікальних best_iteration серед {N_REPEATS_DETERMINISTIC} "
                  f"({sorted(best_iters)}). Нагадування: навіть стабільний результат тут НЕ "
                  f"доводить збіг з оригінальним Step 5D.2 -- лише внутрішню стабільність цієї "
                  f"експериментальної конфігурації.")


Пропущено -- усі датасети вже точно узгоджені (розділ 7).


## 8. Незалежна перевірка CIC-F0 "P/R/F1 = 0 при AP = 0.71" через РЕАЛЬНИЙ розподіл test-скорів

Це та сама аномалія, яку раніше пояснили "артефактом ранньої зупинки" (`best_iteration=4`).
Тут це перевіряється не поясненням, а фактичним розподілом скорів, щойно згенерованих у
розділі 6 (а не переказом попереднього прогону).

In [20]:
scores_cic_f0 = raw_test_scores[("CIC", "F0")]["scores"]
y_cic_f0 = raw_test_scores[("CIC", "F0")]["y_true"]

print("CIC-F0 test-score розподіл (незалежно згенерований у розділі 6):")
print(f"  min={scores_cic_f0.min():.6f}  max={scores_cic_f0.max():.6f}")
for q in [0.5, 0.9, 0.99, 0.999, 0.9999, 1.0]:
    print(f"  quantile {q:.4%}: {np.quantile(scores_cic_f0, q):.6f}")

n_above_half = int((scores_cic_f0 >= 0.5).sum())
print(f"\n  Кількість test-скорів >= 0.5: {n_above_half} з {len(scores_cic_f0):,}")

if n_above_half == 0:
    print(
        "  ✅ Підтверджено НАПРЯМУ (не лише пояснено): жоден test-скор CIC-F0 не досягає "
        "порогу 0.5 -- це узгоджується з дуже раннім best_iteration та НЕ є помилкою "
        "розрахунку Precision/Recall/F1=0 при threshold=0.5. Модель зберігає корисний "
        "ranking (AP=0.71, Top-K precision=1.0), але її абсолютна ймовірнісна шкала при "
        "лише кількох ітераціях бустингу залишається стиснутою нижче 0.5."
    )
else:
    print(
        f"  ❌ НЕОЧІКУВАНО: {n_above_half} скорів >= 0.5, але збережений test_precision/recall/f1 "
        f"вказують на 0 позитивних передбачень -- це суперечність, яку треба розслідувати "
        f"(можлива розбіжність між прогонами)."
    )


CIC-F0 test-score розподіл (незалежно згенерований у розділі 6):
  min=0.110266  max=0.305807
  quantile 50.0000%: 0.110498
  quantile 90.0000%: 0.160341
  quantile 99.0000%: 0.305807
  quantile 99.9000%: 0.305807
  quantile 99.9900%: 0.305807
  quantile 100.0000%: 0.305807

  Кількість test-скорів >= 0.5: 0 з 429,140
  ✅ Підтверджено НАПРЯМУ (не лише пояснено): жоден test-скор CIC-F0 не досягає порогу 0.5 -- це узгоджується з дуже раннім best_iteration та НЕ є помилкою розрахунку Precision/Recall/F1=0 при threshold=0.5. Модель зберігає корисний ranking (AP=0.71, Top-K precision=1.0), але її абсолютна ймовірнісна шкала при лише кількох ітераціях бустингу залишається стиснутою нижче 0.5.


## 9. Збереження артефактів цього аудиту (включно з сирими test-прогнозами для майбутніх повністю артефакт-базованих перевірок)

На відміну від оригінального Step 5D.2, тут додатково зберігаються сирі test-скори,
істинні мітки та індекси рядків — щоб наступного разу перевірку можна було зробити БЕЗ
повторного навчання моделі.

In [21]:
AUDIT_OUT_DIR = RESULTS_DIR / "step5d2_audit"
AUDIT_OUT_DIR.mkdir(exist_ok=True)

reproduced_df.reset_index().to_csv(AUDIT_OUT_DIR / "step5d2_audit_reproduced_results.csv", index=False)

# --- feature_contract.json: явний ПОРЯДОК ознак + hash, окремо для кожної з 4 моделей ---
# (Step 5D.2e/5D.2f forensic-рекомендація: "F0 has 41 features" недостатньо для
# відтворюваності; потрібен "F0 has exactly this ORDERED N-feature contract, hash = ...")
feature_contracts = []
for dataset_key in ["CIC", "UNSW"]:
    for feature_set_name in ["F0", "F1"]:
        feature_order = FEATURE_SETS[(dataset_key, feature_set_name)]
        contract_str = "|".join(feature_order)
        contract_hash = hashlib.sha256(contract_str.encode("utf-8")).hexdigest()
        feature_contracts.append(dict(
            dataset=dataset_key, feature_set=feature_set_name,
            n_features=len(feature_order), feature_order=feature_order,
            feature_order_hash=contract_hash,
        ))
        with open(AUDIT_OUT_DIR / f"feature_contract_{dataset_key}_{feature_set_name}.json", "w") as f:
            json.dump(feature_contracts[-1], f, indent=2)

print("feature_contract_*.json (явний порядок + hash) збережено для всіх 4 моделей:")
for fc in feature_contracts:
    print(f"  {fc['dataset']}-{fc['feature_set']}: n={fc['n_features']}  hash={fc['feature_order_hash'][:16]}...")

for (dataset_key, feature_set_name), payload in raw_test_scores.items():
    out = pd.DataFrame({
        "row_index": payload["index"],
        "y_true": payload["y_true"],
        "test_score": payload["scores"],
    })
    out.to_parquet(AUDIT_OUT_DIR / f"step5d2_audit_raw_test_scores_{dataset_key}_{feature_set_name}.parquet", index=False)

def _stability_to_json(stability_dict):
    return {f"{k[0]}_{k[1]}": [dict(best_iteration=int(bi), test_ap=float(ap)) for bi, ap in runs]
            for k, runs in stability_dict.items()}

summary = dict(
    all_metrics_matched_within_tolerance=bool(all_match),
    dataset_all_match={k: bool(v) for k, v in dataset_all_match.items()},
    delta_ap_match={k: bool(v) for k, v in delta_ap_match.items()},
    ap_tolerance=AP_TOL,
    cic_f0_zero_scores_above_half_confirmed=bool(n_above_half == 0),
    env_info=env_info,
    training_config=dict(
        lgb_params=LGB_PARAMS, num_boost_round=NUM_BOOST_ROUND,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS, random_state=RANDOM_STATE,
    ),
    data_hashes={
        k: {kk: (vv if isinstance(vv, str) else int(vv)) for kk, vv in v.items()}
        for k, v in data_hashes.items()
    },
    repeat_stability=_stability_to_json(repeat_stability),
    deterministic_stability=_stability_to_json(deterministic_stability),
)
with open(AUDIT_OUT_DIR / "step5d2_audit_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Збережено в {AUDIT_OUT_DIR}:")
for p in sorted(AUDIT_OUT_DIR.glob("step5d2_audit_*")):
    print(f"  {p.name}")


feature_contract_*.json (явний порядок + hash) збережено для всіх 4 моделей:
  CIC-F0: n=77  hash=8fccf4905727af2a...
  CIC-F1: n=84  hash=5bc752fa90fce986...
  UNSW-F0: n=41  hash=9e9afca8673a889f...
  UNSW-F1: n=46  hash=95bd664bcc892039...
Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results/step5d2_audit:
  step5d2_audit_raw_test_scores_CIC_F0.parquet
  step5d2_audit_raw_test_scores_CIC_F1.parquet
  step5d2_audit_raw_test_scores_UNSW_F0.parquet
  step5d2_audit_raw_test_scores_UNSW_F1.parquet
  step5d2_audit_reproduced_results.csv
  step5d2_audit_summary.json


## 10. Підсумок аудиту

Заповнюється фактичним виводом після запуску: чи пройшли розділи 2-4 (internal-consistency
на збережених CSV); чи збіглося незалежне відтворення (розділ 7) з server-результатами
Step 5D.2 у межах допуску `1e-4` -- **окремо для CIC і окремо для UNSW**, а не одним спільним
твердженням; якщо для якогось датасету виявлено розбіжність -- який висновок дали
forensic-розділи 7a-7d (версії середовища, hash вхідних даних, repeat-stability з frozen і
з експериментальним deterministic-режимом); і чи підтверджено природу CIC-F0 аномалії
(розділ 8) на щойно згенерованих, а не переказаних скорах.

Твердження про "підтверджену відтворюваність" правомірне лише для того датасету, для якого
розділ 7 фактично дав ✅ по обох feature-set (F0 і F1). Шаблон формулювання для рукопису:

- **Якщо датасет збігся:** "<Dataset>: незалежне відтворення F0 та F1 узгоджується зі
  збереженими результатами Step 5D.2 у межах встановлених допусків; результати <Dataset>
  можуть вважатися незалежно відтвореними за frozen experimental protocol."
- **Якщо датасет НЕ збігся:** "<Dataset>: незалежне відтворення frozen-протоколу не
  відтворило збережені результати Step 5D.2. Feature contract і temporal split відтворені,
  однак результати навчання та best_iteration відрізняються. Тому збережені <Dataset>
  результати не визнаються незалежно відтвореними до встановлення причини розбіжності. Це
  НЕ є доказом помилковості збережених результатів" -- збережений і незалежний прогони
  можуть бути кожен внутрішньо консистентним, але різним, доки причина розбіжності не
  встановлена через розділи 7a-7d.